<a href="https://colab.research.google.com/github/Aadit-5810228/Aadit-5810228-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

First we need to start with glacing over the data for overall spot checks.

In [25]:
#Display the head all the 6 datasets
display(tables["orders"].head())
display(tables["order_status_events"].head())
display(tables["order_lines"].head())
display(tables["payments"].head())
display(tables["restaurants"].head())
display(tables["customers"].head())


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z
3,E021428,O005468,delivered,2026-04-15T13:43:36Z
4,E037134,O009483,picked_up,2026-05-19T12:18:18Z


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5
3,O000002,3,Burrito,3,8.0
4,O000003,1,Fries,1,3.0


,payment_id,order_id,payment_type,amount,paid_at
0,P014735,O013022,charge,88.49,2026-06-18 21:41:37
1,P004116,O003642,charge,52.99,2026-03-31 18:04:12
2,P009951,O008786,charge,67.49,2026-05-13 16:26:11
3,P015198,O013427,charge,83.49,2026-06-21 22:24:53
4,P011527,O010189,charge,21.99,2026-05-25 11:56:06


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds
3,C00004,2024-09-14,London
4,C00005,2024-01-25,Manchester


The output suggests the within the overall data cleaning process we need to check for any NA values, duplicates within the id columns, ensure consistency within the categorical data like the city, cusine or dates and finally spot any major outliers any quantitative data like the delivery fee and amount.

Now we move on to cleaning the data. This process will be done from data set to data set, the data sets will be cleaned as they being checked.

In [32]:
#For the orders data set, check for any NA values within the data set
print(tables["orders"].isna().sum())

#Next also check for any duplicates
print("----------------------")
print(tables["orders"].duplicated().sum())

#Next check the delivery_fee column for outliers using the IQR, and print the outliers
print("----------------------")
Q1 = tables["orders"]["delivery_fee"].quantile(0.25)
Q3 = tables["orders"]["delivery_fee"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = tables["orders"][(tables["orders"]["delivery_fee"] < lower_bound) | (tables["orders"]["delivery_fee"] > upper_bound)]
print(f"IQR: {IQR}, Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")
print(f"Number of delivery fee outliers: {outliers.shape[0]}")

order_id         0
customer_id      0
restaurant_id    0
order_ts         0
delivery_fee     0
dtype: int64
----------------------
0
----------------------
IQR: 1.5000000000000002, Lower Bound: -0.7600000000000005, Upper Bound: 5.24
Number of delivery fee outliers: 0


The orders data seems clean hence we can move on the next data set



In [35]:
#Perform a similar analysis for the NA and dulpicate values on the order_status_events data set
print(tables["order_status_events"].isna().sum())
print("----------------------")
print(tables["order_status_events"].duplicated().sum())

#Check if the formatting within each entry of the status column is consistent
print("----------------------")
print(tables["order_status_events"]["status"].value_counts())

#Check if the date format within the event_ts is consistent
print("----------------------")
try:
    parsed_ts = pd.to_datetime(tables["order_status_events"]["event_ts"], errors='raise')
    print("All event_ts values successfully parsed with default ISO8601 format!")
    print(f"Date Range: {parsed_ts.min()} to {parsed_ts.max()}")
except Exception as e:
    print(f"Parsing failed: {e}")

event_id    0
order_id    0
status      0
event_ts    0
dtype: int64
----------------------
0
----------------------
status
placed       14494
accepted     14154
picked_up    13745
delivered    13549
cancelled      870
Name: count, dtype: int64
----------------------
All event_ts values successfully parsed with default ISO8601 format!
Date Range: 2026-03-01 09:13:23+00:00 to 2026-06-30 23:37:01+00:00


There are no NA values or duplicates within the order_status_events data set. Also the formatting within the status and time stamp columns is consistent

In [39]:
#Perform the NA and dulpicates check on the order_lines data set
print(tables["order_lines"].isna().sum())
print("----------------------")
print(tables["order_lines"].duplicated().sum())

#Check for consistent formatting in the line_no column
print("----------------------")
print(tables["order_lines"]["line_no"].value_counts())

#Check for consistent formatting in the item_name column
print("----------------------")
print(tables["order_lines"]["item_name"].value_counts())

#Check for any outliers in the quantity and unit price columns using the IQR method
print("----------------------")
for col in ["quantity", "unit_price"]:
    Q1 = tables["order_lines"][col].quantile(0.25)
    Q3 = tables["order_lines"][col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = tables["order_lines"][(tables["order_lines"][col] < lower_bound) | (tables["order_lines"][col] > upper_bound)]
    print(f"{col.capitalize()} - IQR: {IQR:.4f}, Lower Bound: {lower_bound:.4f}, Upper Bound: {upper_bound:.4f}")
    print(f"Number of outliers in {col}: {outliers.shape[0]}")
    if outliers.shape[0] > 0:
        display(outliers.head(5))
    print("----------------------")

order_id      0
line_no       0
item_name     0
quantity      0
unit_price    0
dtype: int64
----------------------
0
----------------------
line_no
1    14494
2    10859
3     7179
4     3588
Name: count, dtype: int64
----------------------
item_name
Pad thai         3101
Dessert          3054
Soft drink       3034
Fried rice       3022
Cheeseburger     3016
Burrito          3008
Chicken tikka    3005
Margherita       2994
Fries            2982
Side salad       2976
Ramen            2967
Falafel wrap     2961
Name: count, dtype: int64
----------------------
Quantity - IQR: 2.0000, Lower Bound: -2.0000, Upper Bound: 6.0000
Number of outliers in quantity: 0
----------------------
Unit_price - IQR: 5.5000, Lower Bound: -3.2500, Upper Bound: 18.7500
Number of outliers in unit_price: 0
----------------------


The order_line column is clean as well, there are no NA or duplicate values. The formatting in the line number column is consistent as well. There are no major outliers within the quantitative data.

In [73]:
#Now perform the NA and duplicate value checks for the payments data set. Display the duplicates
print(tables["payments"].isna().sum())
print("----------------------")
print(tables["payments"].duplicated().sum())
duplicate_payments = tables["payments"][tables["payments"].duplicated(keep=False)]
print(f"Total rows involved in duplicates (including originals): {len(duplicate_payments)}")
display(duplicate_payments.sort_values(by="payment_id"))

#Check for consistent formatting in the payment_type column
print("----------------------")
print(tables["payments"]["payment_type"].value_counts())

#Check for consistent formatting in the paid_at column
print("----------------------")
try:
    parsed_ts = pd.to_datetime(tables["payments"]["paid_at"], errors='raise')
    print("All paid_at values successfully parsed with default ISO8601 format!")
    print(f"Date Range: {parsed_ts.min()} to {parsed_ts.max()}")
except Exception as e:
    print(f"Parsing failed: {e}")




payment_id      0
order_id        0
payment_type    0
amount          0
paid_at         0
dtype: int64
----------------------
0
Total rows involved in duplicates (including originals): 0


,payment_id,order_id,payment_type,amount,paid_at


----------------------
payment_type
charge    14494
refund     1899
Name: count, dtype: int64
----------------------
All paid_at values successfully parsed with default ISO8601 format!
Date Range: 2026-03-01 09:14:35 to 2026-07-02 17:13:58


In [ ]:
#There are duplicate entires in the Drop the duplicate entries
tables["payments"].drop_duplicates(inplace=True)

The payments data is cleaned, there are no NA values and all the duplicate values within the data set have been cleaned out. The entries for refund orders are kept for now. But they will be dropped with thier paralel order IDs when moving the data to the gold madalion stage to avoid unecessary data entries during aggregation.

In [60]:
#Now check for NA and duplicates in the restaurants data set
print(tables["restaurants"].isna().sum())
print("----------------------")
print(tables["restaurants"].duplicated().sum())

#Check for consistent formatting in the cuisine and city column
print("----------------------")
print(tables["restaurants"]["cuisine"].value_counts())
print("----------------------")
print(tables["restaurants"]["city"].value_counts())

#Check is the formatting for the data is consistent in the effective_from column
print("----------------------")
try:
    parsed_effective = pd.to_datetime(tables["restaurants"]["effective_from"], errors='raise')
    print("All effective_from values successfully parsed as timestamps!")
    print(f"Date Range: {parsed_effective.min()} to {parsed_effective.max()}")
    print("Unique formats count:")
    print(tables["restaurants"]["effective_from"].apply(len).value_counts())
except Exception as e:
    print(f"Parsing failed: {e}")
print("----------------------")



restaurant_id      0
restaurant_name    0
cuisine            0
city               0
effective_from     0
dtype: int64
----------------------
0
----------------------
cuisine
vegan       9
japanese    9
thai        9
burgers     8
indian      8
italian     8
mexican     8
chinese     8
Name: count, dtype: int64
----------------------
city
london        24
manchester    18
birmingham    12
bristol        7
leeds          6
Name: count, dtype: int64
----------------------
All effective_from values successfully parsed as timestamps!
Date Range: 2025-01-01 00:00:00 to 2026-05-27 00:00:00
Unique formats count:
effective_from
10    67
Name: count, dtype: int64
----------------------


In [57]:
#Make the formatting within the cuisine and city columns consistent by using lower case and stripping spaces
tables["restaurants"]["cuisine"] = tables["restaurants"]["cuisine"].str.lower().str.strip()
tables["restaurants"]["city"] = tables["restaurants"]["city"].str.lower().str.strip()

#Compare the last check of formatting in the cuisine and city column with a fresh one
print("----------------------")
print("Fresh verification of consistent lowercase and stripped formatting:")
print(tables["restaurants"]["cuisine"].value_counts())
print("----------------------")
print(tables["restaurants"]["city"].value_counts())

----------------------
Fresh verification of consistent lowercase and stripped formatting:
cuisine
vegan       9
japanese    9
thai        9
burgers     8
indian      8
italian     8
mexican     8
chinese     8
Name: count, dtype: int64
----------------------
city
london        24
manchester    18
birmingham    12
bristol        7
leeds          6
Name: count, dtype: int64


There are no NA values present int he restaurants data set. The formatting for the data within the effective_from column have also been checked. There were inconsistencies within cuisine and city column which have been cleaned to make it consistent.

In [63]:
#Check for NA and duplicate values in the customers data set
print(tables["customers"].isna().sum())
print("----------------------")
print(tables["customers"].duplicated().sum())

#Check for consistent formatting within the signup_date column
print("----------------------")
try:
    parsed_signup = pd.to_datetime(tables["customers"]["signup_date"], errors='raise')
    print("All signup_date values successfully parsed as timestamps!")
    print(f"Date Range: {parsed_signup.min()} to {parsed_signup.max()}")
    print("Unique length configurations:")
    print(tables["customers"]["signup_date"].apply(len).value_counts())
except Exception as e:
    print(f"Parsing failed: {e}")

#Check formatting in the home_city column
print("----------------------")
print(tables["customers"]["home_city"].value_counts())

customer_id    0
signup_date    0
home_city      0
dtype: int64
----------------------
0
----------------------
All signup_date values successfully parsed as timestamps!
Date Range: 2024-01-02 00:00:00 to 2026-03-10 00:00:00
Unique length configurations:
signup_date
10    4200
Name: count, dtype: int64
----------------------
home_city
London        1541
Manchester     857
Birmingham     697
Bristol        569
Leeds          536
Name: count, dtype: int64


The customers data set is clean. There are no NA values or duplicates. The dates within the signup_date column are also consistent

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

Add text here

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Now we need to merge the data sets to the orders column using the primary keys- order id, customer id and restaurant id


In [68]:
#Use left join to attatch the order_status_events, order_lines and payments to the orders data set on order_id.
orders_merged = tables["orders"].merge(tables["order_status_events"], on="order_id", how="left")
orders_merged = orders_merged.merge(tables["order_lines"], on="order_id", how="left")
orders_merged = orders_merged.merge(tables["payments"], on="order_id", how="left")

#restaurants on restaurant_id to orders
orders_merged = orders_merged.merge(tables["restaurants"], on="restaurant_id", how="left")

#customers on customer_id to orders
orders_merged = orders_merged.merge(tables["customers"], on="customer_id", how="left")

# Display metadata and head of the combined dataset to verify shapes and contents
print(f"Joined dataset shape: {orders_merged.shape}")
display(orders_merged)

Joined dataset shape: (161171, 22)


,order_id,customer_id,restaurant_id,order_ts,delivery_fee,event_id,status,event_ts,line_no,item_name,...,payment_id,payment_type,amount,paid_at,restaurant_name,cuisine,city,effective_from,signup_date,home_city
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49,E000016,delivered,2026-03-01T10:13:39Z,1,Ramen,...,P000001,charge,37.49,2026-03-01 09:14:35,Old Burgers 52,burgers,london,2025-01-01,2025-07-03,Bristol
1,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49,E000001,placed,2026-03-01T09:13:23Z,1,Ramen,...,P000001,charge,37.49,2026-03-01 09:14:35,Old Burgers 52,burgers,london,2025-01-01,2025-07-03,Bristol
2,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49,E000002,accepted,2026-03-01T09:15:28Z,1,Ramen,...,P000001,charge,37.49,2026-03-01 09:14:35,Old Burgers 52,burgers,london,2025-01-01,2025-07-03,Bristol
3,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49,E000009,picked_up,2026-03-01T09:37:00Z,1,Ramen,...,P000001,charge,37.49,2026-03-01 09:14:35,Old Burgers 52,burgers,london,2025-01-01,2025-07-03,Bristol
4,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49,E000003,placed,2026-03-01T09:15:33Z,1,Soft drink,...,P000002,charge,31.99,2026-03-01 09:16:45,Corner Chinese 27,chinese,london,2025-01-01,2024-05-04,Leeds
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161166,O014494,C04040,R058,2026-06-30T22:58:46Z,2.49,E056806,accepted,2026-06-30T23:02:23Z,2,Chicken tikka,...,P016393,charge,59.49,2026-06-30 23:00:02,Grand Indian 58,indian,manchester,2025-01-01,2025-08-02,Leeds
161167,O014494,C04040,R058,2026-06-30T22:58:46Z,2.49,E056810,picked_up,2026-06-30T23:23:26Z,1,Burrito,...,P016393,charge,59.49,2026-06-30 23:00:02,Grand Indian 58,indian,manchester,2025-01-01,2025-08-02,Leeds
161168,O014494,C04040,R058,2026-06-30T22:58:46Z,2.49,E056810,picked_up,2026-06-30T23:23:26Z,2,Chicken tikka,...,P016393,charge,59.49,2026-06-30 23:00:02,Grand Indian 58,indian,manchester,2025-01-01,2025-08-02,Leeds
161169,O014494,C04040,R058,2026-06-30T22:58:46Z,2.49,E056812,delivered,2026-06-30T23:37:01Z,1,Burrito,...,P016393,charge,59.49,2026-06-30 23:00:02,Grand Indian 58,indian,manchester,2025-01-01,2025-08-02,Leeds


Now I will check if any data entry was lost in the process of merging the data set

In [69]:
#Do a check if any data entry was lost in the process of merging by comparing it back to the individual data sets
print(f"Original orders: {tables['orders'].shape[0]} rows")
print(f"Original order status events: {tables['order_status_events'].shape[0]} rows")
print(f"Original order lines: {tables['order_lines'].shape[0]} rows")
print(f"Original payments: {tables['payments'].shape[0]} rows")
print(f"Merged dataset: {orders_merged.shape[0]} rows")

# Ensure unique order count matches the original source exactly
print("----------------------")
print(f"Unique order IDs in source orders: {tables['orders']['order_id'].nunique()}")
print(f"Unique order IDs in merged dataset: {orders_merged['order_id'].nunique()}")


Original orders: 14494 rows
Original order status events: 56812 rows
Original order lines: 36120 rows
Original payments: 16393 rows
Merged dataset: 161171 rows
----------------------
Unique order IDs in source orders: 14494
Unique order IDs in merged dataset: 14494


The number of order id remain consistent hence all the data was mergered successfuly without any loss

In [71]:
# Check for missing values in each column of the merged dataset
print("--- Missing values count per column in merged dataset ---")
missing_counts = orders_merged.isna().sum()
display(missing_counts[missing_counts > 0])
if (missing_counts == 0).all():
    print("Success! No missing values found in any columns of the merged dataset.")

--- Missing values count per column in merged dataset ---


,0


Success! No missing values found in any columns of the merged dataset.


There are no NA values in the data set hence the data sets were cleanly merged as well

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

This was done during the steps above


## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

The comments added with the code are the prompts used for the code

## 6. What I could not verify

Certain steps that I missed out on due to the time crunch include
*   Creating the aggregated data set with the weekly numbers
*   A more descriptive check of individual data entries within the merged data set



Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.